# 02. Qubits, Bra-Ket Notation & Density Matrices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rfandan/computational-quantum-optics/blob/main/module_0_qm_fundamentals/02_qubits_and_density_matrices.ipynb)

* **Module:** 0 — Quantum Mechanics Fundamentals
* **Difficulty:** Beginner
* **Estimated Time:** 40 minutes
* **Prerequisites:** Notebook 0.1 (Continuous Wavefunctions & Qubit Concept), Elementary Linear Algebra (Matrix Multiplication, Trace, Eigenvalues).

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. **Manipulate discrete quantum states** using Dirac bra-ket notation and QuTiP state objects (`basis`, `Qobj`).
2. **Perform operator algebra** with the Pauli matrices ($\sigma_x, \sigma_y, \sigma_z$) and evaluate expectation values.
3. **Formulate the Density Matrix ($\rho$)** to represent both pure quantum superpositions and classical statistical ensembles.
4. **Compute fundamental quantum invariants:** Trace ($\mathrm{Tr}(\rho)=1$), Purity ($\mathcal{P} = \mathrm{Tr}(\rho^2)$), and von Neumann entropy ($S(\rho)$).
5. **Evaluate the Partial Trace** ($\mathrm{Tr}_B(\rho_{AB})$) on entangled bipartite systems to understand the origin of decoherence.
6. **Map and visualize quantum states on the 3D Bloch Sphere**, distinguishing pure surface states ($r = 1$) from mixed interior states ($r < 1$).


---
## 0. Setup & Dependencies

We will use **QuTiP (Quantum Toolbox in Python)** as our primary computational framework alongside NumPy and Matplotlib.


In [ ]:
# Install QuTiP and dependencies in Google Colab (uncomment if running in Colab)
# !pip install -q qutip numpy scipy matplotlib ipywidgets

import numpy as np
import matplotlib.pyplot as plt

# Import QuTiP core functions
try:
    import qutip as qt
    from qutip import basis, sigmax, sigmay, sigmaz, qeye, tensor
    print(f"QuTiP version {qt.__version__} loaded successfully!")
except ImportError:
    print("QuTiP not found. Please install via: pip install qutip")

# Configure clean, publication-ready plotting aesthetics
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})


---
## 1. Dirac Bra-Ket Notation & Qubits in QuTiP

In 1939, Paul Dirac introduced the elegant **bra-ket notation** that unifies wave mechanics and matrix algebra. 

A quantum state is represented by a **ket vector** $|\psi\rangle$ living in a complex Hilbert space $\mathcal{H} = \mathbb{C}^d$. The dual vector (the conjugate transpose) is represented by a **bra vector** $\langle\psi| = (|\psi\rangle)^\dagger$.

### The Qubit Computational Basis
For a two-level quantum system (such as the ground and excited states of an atom $|g\rangle, |e\rangle$ or horizontal and vertical photon polarizations $|H\rangle, |V\rangle$), the Hilbert space is $\mathbb{C}^2$. The orthonormal computational basis states are:

$$|0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \quad |1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$$

Any arbitrary normalized pure state is a linear combination:

$$|\psi\rangle = \alpha |0\rangle + \beta |1\rangle = \begin{pmatrix} \alpha \\ \beta \end{pmatrix}, \quad \langle\psi|\psi\rangle = |\alpha|^2 + |\beta|^2 = 1$$

In QuTiP, a state vector or operator is represented as a `Qobj` (Quantum Object). We generate basis vectors using `basis(dimension, state_index)`.


In [ ]:
# Generate standard computational basis kets in QuTiP
psi0 = basis(2, 0)  # |0>
psi1 = basis(2, 1)  # |1>

print("|0> state:")
display(psi0) if 'display' in globals() else print(psi0)

print("\n|1> state:")
display(psi1) if 'display' in globals() else print(psi1)

# Inner product: <0|1> (orthonormal basis)
overlap_01 = psi0.overlap(psi1)  # or (psi0.dag() * psi1).tr()
print(f"Overlap <0|1> = {overlap_01}")

# Create superposition states: |+> and |->
psi_plus = (psi0 + psi1).unit()   # (|0> + |1>) / sqrt(2)
psi_minus = (psi0 - psi1).unit()  # (|0> - |1>) / sqrt(2)

print("\n|+> state (normalized):")
print(psi_plus)

# Outer product: Projection operator P0 = |0><0|
P0 = psi0 * psi0.dag()
print("\nProjection Operator |0><0|:")
print(P0)


> **Key Concept:** Notice the difference between the **inner product** $\langle \phi | \psi \rangle$ (a complex scalar representing probability amplitude) and the **outer product** $|\psi\rangle\langle\phi|$ (a matrix representing an operator or projector).


---
## 2. Observables & The Pauli Spin Algebra

Physical observables for a qubit are represented by $2 \times 2$ Hermitian matrices. The three **Pauli operators** together with the identity matrix $\mathbb{I}_2$ form a complete basis for all $2 \times 2$ complex matrices:

$$\sigma_x = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \quad \sigma_y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}, \quad \sigma_z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}, \quad \mathbb{I} = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}$$

### Algebraic Properties
1. **Involutory (Self-inverse):** $\sigma_x^2 = \sigma_y^2 = \sigma_z^2 = \mathbb{I}$
2. **Commutation relations:** $[\sigma_i, \sigma_j] = 2i \epsilon_{ijk} \sigma_k$ (e.g., $[\sigma_x, \sigma_y] = 2i \sigma_z$)
3. **Anti-commutation relations:** $\{\sigma_i, \sigma_j\} \equiv \sigma_i \sigma_j + \sigma_j \sigma_i = 2\delta_{ij}\mathbb{I}$

In quantum optics:
* $\sigma_z$ measures atomic energy inversion (difference between excited and ground state populations).
* $\sigma_x$ and $\sigma_y$ govern electric dipole transitions (interaction with the laser field).
* $\sigma_+ = \frac{1}{2}(\sigma_x + i\sigma_y) = |1\rangle\langle 0|$ acts as the atomic raising operator.
* $\sigma_- = \frac{1}{2}(\sigma_x - i\sigma_y) = |0\rangle\langle 1|$ acts as the atomic lowering operator.


In [ ]:
# Initialize Pauli matrices in QuTiP
sx = sigmax()
sy = sigmay()
sz = sigmaz()
id2 = qeye(2)

# Verify the fundamental commutation relation: [sigma_x, sigma_y] = 2i * sigma_z
commutator_xy = sx * sy - sy * sx
expected_commutator = 2j * sz

print("Commutator [sx, sy]:")
print(commutator_xy)
print(f"Matches 2i * sz? {commutator_xy == expected_commutator}")

# Atomic transition operators: sigma_+ and sigma_-
sig_plus = (sx + 1j * sy) / 2.0
sig_minus = (sx - 1j * sy) / 2.0

print("\nAtomic Raising Operator sigma_+ (|1><0|):")
print(sig_plus)

# Test action on ground state |0>: sigma_+ |0> = |1>
state_excited = sig_plus * psi0
print(f"sigma_+ |0> equals |1>? {state_excited == psi1}")


---
## 3. Density Matrices: Pure States vs. Statistical Mixtures

Why do we need density matrices? State vectors $|\psi\rangle$ can only describe systems about which we have **complete quantum information** (pure states). 

However, in realistic experiments:
1. **Classical Ignorance:** An optical source might emit state $|0\rangle$ with probability $50\%$ and state $|1\rangle$ with probability $50\%$ due to thermal or experimental fluctuations.
2. **Entanglement with an Unmeasured Environment:** When a quantum system interacts with a bath or a cavity mode that leaks away, the remaining subsystem cannot be described by any single state vector $|\psi\rangle$.

### The Density Operator Formulation
Lev Landau and John von Neumann independently formulated the **density operator** $\rho$.

* For a **Pure State** $|\psi\rangle$:
  $$\rho_{\text{pure}} = |\psi\rangle\langle\psi|$$

* For a **Mixed State** (statistical ensemble of states $|\psi_i\rangle$ prepared with classical probabilities $p_i$):
  $$\rho_{\text{mixed}} = \sum_i p_i |\psi_i\rangle\langle\psi_i|, \quad \sum_i p_i = 1, \quad 0 \le p_i \le 1$$

### Coherent Superposition vs. Incoherent Mixture
Consider the crucial difference between:
1. **Coherent Superposition:** $|\psi\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$
   $$\rho_{\text{sup}} = \begin{pmatrix} 1/\sqrt{2} \\ 1/\sqrt{2} \end{pmatrix} \begin{pmatrix} 1/\sqrt{2} & 1/\sqrt{2} \end{pmatrix} = \begin{pmatrix} 0.5 & \mathbf{0.5} \\ \mathbf{0.5} & 0.5 \end{pmatrix}$$
2. **Incoherent Statistical Mixture:** $50\%$ chance of $|0\rangle$, $50\%$ chance of $|1\rangle$
   $$\rho_{\text{mix}} = 0.5 |0\rangle\langle 0| + 0.5 |1\rangle\langle 1| = \begin{pmatrix} 0.5 & \mathbf{0} \\ \mathbf{0} & 0.5 \end{pmatrix}$$

The diagonal entries $\rho_{00}, \rho_{11}$ are identical (**populations**). But the off-diagonal terms $\rho_{01}, \rho_{10}$ (**quantum coherences**) are non-zero *only* for the superposition!


In [ ]:
# 1. Coherent Superposition Density Matrix
psi_sup = (basis(2, 0) + basis(2, 1)).unit()
rho_sup = psi_sup * psi_sup.dag()

# 2. Incoherent Mixture Density Matrix
rho_mix = 0.5 * (basis(2, 0) * basis(2, 0).dag()) + 0.5 * (basis(2, 1) * basis(2, 1).dag())

print("Coherent Superposition Density Matrix rho_sup:")
print(rho_sup.full())

print("\nIncoherent Statistical Mixture rho_mix:")
print(rho_mix.full())

# Visualize Matrix Elements using Heatmaps
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

im1 = ax1.imshow(np.real(rho_sup.full()), cmap="Blues", vmin=0, vmax=0.6)
ax1.set_title("Coherent Superposition $\\rho_{\\mathrm{sup}}$\n(Coherences Present)", fontsize=11)
ax1.set_xticks([0, 1], labels=[r"$|0\rangle$", r"$|1\rangle$"])
ax1.set_yticks([0, 1], labels=[r"$|0\rangle$", r"$|1\rangle$"])
for (i, j), z in np.ndenumerate(np.real(rho_sup.full())):
    ax1.text(j, i, f"{z:.2f}", ha="center", va="center", color="black" if z < 0.4 else "white", fontweight="bold")

im2 = ax2.imshow(np.real(rho_mix.full()), cmap="Blues", vmin=0, vmax=0.6)
ax2.set_title("Incoherent Mixture $\\rho_{\\mathrm{mix}}$\n(Coherences Vanished)", fontsize=11)
ax2.set_xticks([0, 1], labels=[r"$|0\rangle$", r"$|1\rangle$"])
ax2.set_yticks([0, 1], labels=[r"$|0\rangle$", r"$|1\rangle$"])
for (i, j), z in np.ndenumerate(np.real(rho_mix.full())):
    ax2.text(j, i, f"{z:.2f}", ha="center", va="center", color="black" if z < 0.4 else "white", fontweight="bold")

plt.tight_layout()
plt.show()


> **Physical Insight:** Decoherence in open quantum systems (caused by spontaneous emission or cavity photon leakage) is precisely the physical process that drives the off-diagonal coherences $\rho_{01} \to 0$, turning a quantum superposition into an incoherent classical mixture.


---
## 4. Fundamental Invariants: Trace, Purity, Entropy & The Bloch Sphere

Any valid physical density matrix $\rho$ must satisfy three mathematical conditions:
1. **Hermiticity:** $\rho^\dagger = \rho$ (observable measurement probabilities are real).
2. **Unit Trace:** $\mathrm{Tr}(\rho) = 1$ (total probability sums to unity).
3. **Positive Semi-definiteness:** $\langle\phi|\rho|\phi\rangle \ge 0$ for all $|\phi\rangle$ (eigenvalues $\lambda_i \ge 0$).

### Quantitative Measures of Mixedness
To determine whether a state is pure or mixed, we use:

1. **Purity $\mathcal{P}$:**
   $$\mathcal{P} \equiv \mathrm{Tr}(\rho^2)$$
   * For any **pure state**: $\rho^2 = \rho \implies \mathcal{P} = 1$.
   * For a **mixed state** in dimension $d$: $\frac{1}{d} \le \mathcal{P} < 1$.
   * For a qubit ($d=2$), a maximally mixed state has $\mathcal{P} = \frac{1}{2}$.

2. **Von Neumann Entropy $S(\rho)$:**
   $$S(\rho) \equiv -\mathrm{Tr}(\rho \log_2 \rho) = -\sum_i \lambda_i \log_2 \lambda_i$$
   * For a pure state: $S = 0$ (zero statistical uncertainty).
   * For a maximally mixed qubit: $S = \log_2(2) = 1$ bit.

### The Bloch Vector Representation
Because the Pauli matrices and identity form a basis for $2 \times 2$ matrices, any single-qubit density matrix can be uniquely parameterized by a real 3D vector $\vec{r} = (r_x, r_y, r_z)$ called the **Bloch vector**:

$$\rho = \frac{1}{2}\left( \mathbb{I} + \vec{r}\cdot\vec{\sigma} \right) = \frac{1}{2}\begin{pmatrix} 1 + r_z & r_x - i r_y \\ r_x + i r_y & 1 - r_z \end{pmatrix}$$

The components of $\vec{r}$ are simply the physical expectation values of the Pauli operators:
$$r_x = \mathrm{Tr}(\rho \sigma_x) = \langle\sigma_x\rangle, \quad r_y = \mathrm{Tr}(\rho \sigma_y) = \langle\sigma_y\rangle, \quad r_z = \mathrm{Tr}(\rho \sigma_z) = \langle\sigma_z\rangle$$

The length of the Bloch vector $|\vec{r}|$ directly relates to purity:
$$|\vec{r}|^2 = r_x^2 + r_y^2 + r_z^2 = 2\,\mathrm{Tr}(\rho^2) - 1 = 2\mathcal{P} - 1$$

* **Pure states:** $|\vec{r}| = 1 \implies$ Lie on the **surface** of the Bloch sphere.
* **Mixed states:** $|\vec{r}| < 1 \implies$ Lie inside the **interior** of the Bloch sphere.
* **Maximally mixed state:** $|\vec{r}| = 0 \implies$ Lies at the exact **center** of the sphere ($\vec{r} = (0,0,0)$).


In [ ]:
def analyze_density_matrix(rho: qt.Qobj, name: str = "State") -> dict:
    """
    Computes Trace, Purity, von Neumann Entropy, and Bloch vector components for a qubit density matrix.
    """
    tr_val = rho.tr()
    purity = (rho * rho).tr()
    entropy = qt.entropy_vn(rho, base=2)
    
    # Calculate Bloch vector components r_i = Tr(rho * sigma_i)
    rx = (rho * sigmax()).tr().real
    ry = (rho * sigmay()).tr().real
    rz = (rho * sigmaz()).tr().real
    r_length = np.sqrt(rx**2 + ry**2 + rz**2)
    
    print(f"=== Analysis for: {name} ===")
    print(f"Trace Tr(ρ)       = {tr_val:.4f}")
    print(f"Purity Tr(ρ²)     = {purity:.4f} (Pure=1.0, Max Mixed=0.5)")
    print(f"Entropy S(ρ)      = {entropy:.4f} bits")
    print(f"Bloch vector r  = ({rx:.3f}, {ry:.3f}, {rz:.3f}) | length |r| = {r_length:.3f}\n")
    
    return {"rx": rx, "ry": ry, "rz": rz, "purity": purity, "entropy": entropy}

stats_sup = analyze_density_matrix(rho_sup, "Coherent Superposition |+><+|")
stats_mix = analyze_density_matrix(rho_mix, "Incoherent Mixture 0.5|0><0| + 0.5|1><1|")


In [ ]:
# 3D Bloch Sphere Visualization
# We draw the Bloch sphere and plot the state vectors

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

# Draw wireframe sphere
u = np.linspace(0, 2 * np.pi, 30)
v = np.linspace(0, np.pi, 20)
xs = np.outer(np.cos(u), np.sin(v))
ys = np.outer(np.sin(u), np.sin(v))
zs = np.outer(np.ones(np.size(u)), np.cos(v))
ax.plot_wireframe(xs, ys, zs, color="gray", alpha=0.15, lw=0.8)

# Draw Coordinate Axes
ax.plot([-1.2, 1.2], [0, 0], [0, 0], 'k--', lw=1, alpha=0.5)
ax.plot([0, 0], [-1.2, 1.2], [0, 0], 'k--', lw=1, alpha=0.5)
ax.plot([0, 0], [0, 0], [-1.2, 1.2], 'k--', lw=1, alpha=0.5)

# Axis Labels
ax.text(1.3, 0, 0, r"$x$ ($\sigma_x$)", color="black", fontsize=11)
ax.text(0, 1.3, 0, r"$y$ ($\sigma_y$)", color="black", fontsize=11)
ax.text(0, 0, 1.3, r"$|0\rangle$ ($+z$)", color="black", fontsize=11)
ax.text(0, 0, -1.4, r"$|1\rangle$ ($-z$)", color="black", fontsize=11)

# Plot Pure state vector |+> (Points along +x on surface)
ax.quiver(0, 0, 0, stats_sup["rx"], stats_sup["ry"], stats_sup["rz"],
          color="#1f77b4", lw=3.0, arrow_length_ratio=0.1, label=r"Pure Superposition $|+\rangle$ ($|\vec{r}|=1.0$)")

# Plot Mixed state vector (Points at origin)
ax.scatter([stats_mix["rx"]], [stats_mix["ry"]], [stats_mix["rz"]],
           color="#d62728", s=100, label=r"Maximally Mixed $\rho_{\mathrm{mix}}$ ($|\vec{r}|=0.0$)")

ax.set_box_aspect([1, 1, 1])
ax.set_title("The Bloch Sphere: Pure Surface vs. Mixed Interior States", pad=20)
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.95))
plt.show()


---
## 5. Composite Systems & Entanglement-Induced Decoherence

How does a pure quantum state turn into a mixed state without any classical noise? The answer lies in **entanglement** and the **Partial Trace** ($\mathrm{Tr}_B$).

Consider a bipartite system composed of subsystem $A$ (e.g., an atom) and subsystem $B$ (e.g., a cavity mode or the environment). The combined Hilbert space is the tensor product $\mathcal{H}_A \otimes \mathcal{H}_B$.

Let the composite system be in the maximally entangled **Bell state** $|\Phi^+\rangle$:

$$|\Phi^+\rangle = \frac{1}{\sqrt{2}}\left( |0\rangle_A \otimes |0\rangle_B + |1\rangle_A \otimes |1\rangle_B \right) = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$

The global density matrix $\rho_{AB} = |\Phi^+\rangle\langle\Phi^+|$ is **strictly pure**:
$$\mathrm{Tr}(\rho_{AB}^2) = 1, \quad S(\rho_{AB}) = 0$$

If an observer has access *only* to subsystem $A$ (tracing out environment $B$):

$$\rho_A = \mathrm{Tr}_B(\rho_{AB}) = \sum_{k \in \{0, 1\}} (\mathbb{I}_A \otimes \langle k|_B)\,\rho_{AB}\,(\mathbb{I}_A \otimes |k\rangle_B)$$

Evaluating this partial trace yields:

$$\rho_A = \frac{1}{2}|0\rangle\langle 0| + \frac{1}{2}|1\rangle\langle 1| = \begin{pmatrix} 0.5 & 0 \\ 0 & 0.5 \end{pmatrix} = \frac{1}{2}\mathbb{I}_2$$

Subsystem $A$ is in a **maximally mixed state** with purity $\mathcal{P}_A = 0.5$ and maximum entropy $S_A = 1$ bit!


In [ ]:
# Construct composite Bell state using tensor products in QuTiP
psi_00 = tensor(basis(2, 0), basis(2, 0))  # |00>
psi_11 = tensor(basis(2, 1), basis(2, 1))  # |11>

# Entangled Bell State |Phi+> = (|00> + |11>) / sqrt(2)
psi_bell = (psi_00 + psi_11).unit()
rho_AB = psi_bell * psi_bell.dag()

# Evaluate Global Bipartite System AB
purity_global = (rho_AB * rho_AB).tr()
entropy_global = qt.entropy_vn(rho_AB, base=2)

print("--- Global Bipartite System (A + B) ---")
print(f"Global Purity Tr(\u03c1_AB\u00b2) = {purity_global:.4f} (Pure!)")
print(f"Global Entropy S(\u03c1_AB)  = {entropy_global:.4f} bits")

# Compute Partial Trace over Subsystem B to obtain Reduced Density Matrix rho_A
rho_A = rho_AB.ptrace(0)  # Trace out subsystem index 1 (B), keep index 0 (A)

purity_local = (rho_A * rho_A).tr()
entropy_local = qt.entropy_vn(rho_A, base=2)

print("\n--- Reduced Subsystem A (Partial Trace over B) ---")
print("Reduced Density Matrix \u03c1_A:")
print(rho_A.full())
print(f"Subsystem A Purity Tr(\u03c1_A\u00b2) = {purity_local:.4f} (Maximally Mixed!)")
print(f"Subsystem A Entropy S(\u03c1_A)  = {entropy_local:.4f} bits")


> **Physical Insight:** This provides the statistical foundation for **Open Quantum Systems (Module 4)**. Whenever a quantum system interacts and becomes entangled with an unobserved environment, local quantum coherence leaks into non-local correlations, causing the subsystem to experience decoherence.


---
## 6. Interactive Playground: Depolarizing & Dephasing Dynamics

In quantum optics and quantum information, interactions with an environment cause state decay. Two canonical noise channels are:
1. **Depolarizing Noise:** The state contracts isotropically toward the maximally mixed state:
   $$\rho_{\text{dep}}(p) = (1-p)|\psi\rangle\langle\psi| + p \frac{\mathbb{I}}{2}$$
2. **Dephasing (Phase Damping):** Only off-diagonal coherences decay, preserving populations:
   $$\rho_{\text{deph}}(p) = \begin{pmatrix} \rho_{00} & (1-p)\rho_{01} \\ (1-p)\rho_{10} & \rho_{11} \end{pmatrix}$$

Use the interactive simulation below to observe how noise parameter $p \in [0, 1]$ degrades purity and shrinks the Bloch vector.


In [ ]:
def simulate_decoherence_curves():
    p_vals = np.linspace(0, 1.0, 100)
    
    # Start from pure |+> state
    psi_init = (basis(2, 0) + basis(2, 1)).unit()
    rho_init = psi_init * psi_init.dag()
    
    purity_depol = []
    entropy_depol = []
    bloch_r_depol = []
    
    purity_deph = []
    bloch_r_deph = []
    
    for p in p_vals:
        # Depolarizing channel
        rho_d = (1.0 - p) * rho_init + p * 0.5 * qeye(2)
        purity_depol.append((rho_d * rho_d).tr())
        entropy_depol.append(qt.entropy_vn(rho_d, base=2))
        rx = (rho_d * sigmax()).tr().real
        bloch_r_depol.append(rx)
        
        # Dephasing channel
        rho_dp_mat = rho_init.full().copy()
        rho_dp_mat[0, 1] *= (1.0 - p)
        rho_dp_mat[1, 0] *= (1.0 - p)
        rho_dp = qt.Qobj(rho_dp_mat)
        purity_deph.append((rho_dp * rho_dp).tr())
        rx_dp = (rho_dp * sigmax()).tr().real
        bloch_r_deph.append(rx_dp)
        
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    
    # 1. Purity & Entropy under Depolarization
    ax1.plot(p_vals, purity_depol, label=r"Purity $\mathcal{P} = \mathrm{Tr}(\rho^2)$", color="#1f77b4", lw=2)
    ax1.plot(p_vals, entropy_depol, label=r"Entropy $S(\rho)$ [bits]", color="#d62728", lw=2, ls="--")
    ax1.axhline(0.5, color="gray", ls=":", label="Minimum Purity (0.5)")
    ax1.set_title("Depolarizing Noise: Purity vs. Entropy")
    ax1.set_xlabel("Noise Probability $p$")
    ax1.set_ylabel("Metric Value")
    ax1.legend(loc="center left")
    
    # 2. Bloch Vector Length Decay
    ax2.plot(p_vals, bloch_r_depol, label=r"Depolarizing ($|\vec{r}| \to 0$)", color="#1f77b4", lw=2)
    ax2.plot(p_vals, bloch_r_deph, label=r"Pure Dephasing ($r_x \to 0$)", color="#2ca02c", lw=2, ls="-.")
    ax2.set_title("Bloch Vector Contraction Under Noise")
    ax2.set_xlabel("Noise Parameter $p$")
    ax2.set_ylabel("Bloch Vector Component $r_x$")
    ax2.legend(loc="upper right")
    
    plt.tight_layout()
    plt.show()

simulate_decoherence_curves()


---
## 7. Hands-On Exercises & Guided Challenges

### Exercise 1: The Werner State & Entanglement Threshold
The **Werner state** is a mixture of the maximally entangled Bell state $|\Phi^+\rangle$ and complete unpolarized noise:

$$\rho_W(p) = p\,|\Phi^+\rangle\langle\Phi^+| + \frac{1-p}{4}\,\mathbb{I}_4 \quad (0 \le p \le 1)$$

1. Compute the purity $\mathcal{P}(p) = \mathrm{Tr}(\rho_W^2)$ as an analytical function of $p$.
2. Implement $\rho_W(p)$ in QuTiP and plot $\mathcal{P}$ for $p \in [0, 1]$. What is the purity when $p=0$ (pure noise)?

<details>
<summary><b>Click for Hint & Solution</b></summary>

**Analytical derivation:**
$$\rho_W^2 = p^2 (|\Phi^+\rangle\langle\Phi^+|)^2 + \left(\frac{1-p}{4}\right)^2 \mathbb{I}_4^2 + 2p\left(\frac{1-p}{4}\right)|\Phi^+\rangle\langle\Phi^+|$$
Taking the trace using $\mathrm{Tr}(|\Phi^+\rangle\langle\Phi^+|) = 1$ and $\mathrm{Tr}(\mathbb{I}_4) = 4$:
$$\mathrm{Tr}(\rho_W^2) = p^2 + \frac{(1-p)^2}{4} + \frac{p(1-p)}{2} = \frac{1 + 3p^2}{4}$$
* At $p=1$ (Bell state): $\mathcal{P} = 1.0$.
* At $p=0$ (Maximally mixed state in dimension 4): $\mathcal{P} = 1/4 = 0.25$.

**Code snippet:**
```python
p_grid = np.linspace(0, 1, 50)
purities = []
id4 = qt.tensor(qeye(2), qeye(2))
for p in p_grid:
    rho_w = p * rho_AB + (1.0 - p) / 4.0 * id4
    purities.append((rho_w * rho_w).tr())

plt.figure(figsize=(7, 4))
plt.plot(p_grid, purities, 'b-', lw=2, label=r"Numerical $\mathrm{Tr}(\rho_W^2)$")
plt.plot(p_grid, (1 + 3*p_grid**2)/4, 'r--', label=r"Analytical $(1 + 3p^2)/4$")
plt.xlabel("Parameter $p$")
plt.ylabel("Purity")
plt.legend()
plt.show()
```
</details>

---

### Exercise 2: State Tomography via Pauli Measurements
Suppose an unknown qubit state $\rho$ is measured in three orthogonal measurement bases, yielding:
$$\langle \sigma_x \rangle = 0.6, \quad \langle \sigma_y \rangle = -0.4, \quad \langle \sigma_z \rangle = 0.5$$

1. Calculate the Bloch vector length $|\vec{r}|$. Is this state pure or mixed?
2. Reconstruct the full $2 \times 2$ density matrix $\rho$ using the Bloch decomposition formula.
3. Check its eigenvalues to ensure it is positive semi-definite.

<details>
<summary><b>Click for Hint & Solution</b></summary>

**Solution:**
1. $|\vec{r}|^2 = 0.6^2 + (-0.4)^2 + 0.5^2 = 0.36 + 0.16 + 0.25 = 0.77 < 1.0$.
   Since $|\vec{r}| = \sqrt{0.77} \approx 0.877 < 1$, the state is **mixed**.
2. Reconstruction:
   $$\rho = \frac{1}{2}\begin{pmatrix} 1 + 0.5 & 0.6 - i(-0.4) \\ 0.6 + i(-0.4) & 1 - 0.5 \end{pmatrix} = \begin{pmatrix} 0.75 & 0.3 + 0.2i \\ 0.3 - 0.2i & 0.25 \end{pmatrix}$$
3. Eigenvalues: $\lambda_\pm = \frac{1 \pm |\vec{r}|}{2} = \frac{1 \pm 0.877}{2} \approx 0.939, 0.061$. Both are non-negative, confirming $\rho$ is a valid physical state!

**Code snippet:**
```python
rx, ry, rz = 0.6, -0.4, 0.5
rho_rec = 0.5 * (qeye(2) + rx*sigmax() + ry*sigmay() + rz*sigmaz())
print("Reconstructed rho:")
print(rho_rec.full())
print(f"Eigenvalues: {rho_rec.eigenenergies()}")
```
</details>


---
## 8. Summary & What's Next

### Key Takeaways
1. **Dirac Bra-Ket Formalism:** Kets $|\psi\rangle$ denote state vectors in Hilbert space $\mathbb{C}^d$; bras $\langle\psi|$ denote dual elements.
2. **Pauli Algebra:** The Pauli matrices ($\sigma_x, \sigma_y, \sigma_z$) represent angular momentum and atomic inversion/transition observables with fundamental commutation relation $[\sigma_i, \sigma_j] = 2i\epsilon_{ijk}\sigma_k$.
3. **Density Matrices ($\rho$):** Describe both pure states ($|\psi\rangle\langle\psi|$) and classical ensembles ($\sum p_i |\psi_i\rangle\langle\psi_i|$). Coherent superpositions feature non-zero off-diagonal coherences; incoherent mixtures do not.
4. **Purity and the Bloch Sphere:** The Bloch vector $\vec{r} = (\langle\sigma_x\rangle, \langle\sigma_y\rangle, \langle\sigma_z\rangle)$ provides a 1-to-1 geometric mapping of qubit states: pure states lie on the surface ($|\vec{r}|=1, \mathcal{P}=1$), mixed states lie in the interior ($|\vec{r}|<1, \mathcal{P}<1$).
5. **Partial Trace & Decoherence:** Tracing out an entangled subsystem produces a mixed state, demonstrating how environmental entanglement destroys local quantum coherence.

---

### What's Next?
In **Notebook 0.3 (`03_observables_measurement_uncertainty.ipynb`)**, we will:
* Implement differential operators on discrete grids ($\hat{x}, \hat{p}$).
* Calculate operator commutators and numerically evaluate $[\hat{x}, \hat{p}] = i\hbar \mathbb{I}$.
* Simulate projective measurements and test the Robertson-Schrödinger uncertainty relation:
  $$\Delta A \Delta B \ge \frac{1}{2} |\langle [\hat{A}, \hat{B}] \rangle|$$

---
### References & Recommended Reading
* **Michael A. Nielsen & Isaac L. Chuang**, *Quantum Computation and Quantum Information*, Chapter 2: Introduction to Quantum Mechanics.
* **Christopher Gerry & Peter Knight**, *Introductory Quantum Optics*, Chapter 1 & 2.
* **QuTiP Documentation**, *Basic Operations on Quantum Objects* (https://qutip.org/docs/latest/).
